#Healthcare Data Lakehouse Pipeline

In [0]:
RAW_PATH='/Volumes/workspace/default/healthcare_data_lake/'
BRONZE_PATH='/Volumes/workspace/default/healthcare_data_lake/bronze/'
SILVER_PATH='/Volumes/workspace/default/healthcare_data_lake/silver/'
GOLD_PATH='/Volumes/workspace/default/healthcare_data_lake/gold/'

#KEY OF GCP PROJECT
GCP_PROJECT='directed-truck-489818-t6'
BQ_QUERY='healthcare'
TEMP_GCS_BUCKET='healthcare-databricks-temp'


#secrets codes
GCP_SECRET_SCOPE='gcp-secret'
GCP_SECRET_KEY='gcp-sa-key'

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import lit

diagnosis_schema=StructType([
    StructField ('diagnosis_id', IntegerType(), True),
    StructField('diagnosis_name', StringType(),True)
])

labs_schema=StructType([
    StructField('lab_id', IntegerType(), True),
    StructField('patient_id', IntegerType(), True),
    StructField('test_name', StringType(), True),
    StructField('result', DoubleType(), True),
    StructField('normal_range', StringType(), True)
])

outcomes_schema=StructType([
    StructField('outcome_id', IntegerType(), True),
    StructField('outcome_name', StringType(), True)
])

patients_schema=StructType([
    StructField('patient_id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('diagnosis_id', IntegerType(), True),
    StructField('admission_date', TimestampType(), True),
    StructField('discharge_date', TimestampType(), True),
    StructField('outcome_id', IntegerType(), True),
    StructField('treatment_cost', DoubleType(), True)
])

diagnosis_df=spark.read.option('header', True).schema(diagnosis_schema).csv(RAW_PATH+'diagnoses.csv')
labs_df=spark.read.option('header',True).schema(labs_schema).csv(RAW_PATH+'labs.csv')
outcomes_df=spark.read.option('header', True).schema(outcomes_schema).csv(RAW_PATH+'outcomes.csv')
patients_df=spark.read.option('header', True).schema(patients_schema).csv(RAW_PATH+'patients.csv')

diagnosis_df.printSchema()
    

root
 |-- diagnosis_id: integer (nullable = true)
 |-- diagnosis_name: string (nullable = true)



In [0]:
patients_df.selectExpr('count(distinct diagnosis_id) as unique_diagnosis_id').display()


unique_diagnosis_id
10


In [0]:
from pyspark.sql.functions import count
labs_df.filter('test_name = "Blood Pressure"').select(count('patient_id')).display()


count(patient_id)
37


In [0]:
diagnosis_df.write.mode('overwrite').format('delta').option('overwriteScham', True).save(BRONZE_PATH+'diagnosis')
labs_df.write.mode('overwrite').format('delta').option('overwriteSchema', True).save(BRONZE_PATH+'labs')
outcomes_df.write.mode('overwrite').format('delta').option('overwriteSchema', True).save(BRONZE_PATH+'outcomes')
patients_df.write.mode('overwrite').format('delta').option('overwriteSchema', True).save(BRONZE_PATH+'patients')
